# inbox

> Braincave inbox triage — read notes, apply Jelle's verdict, keep an undo stack.

Stage 1 of the human-gated inbox pipeline. Jelle makes the keep / delete / too-hard call
himself; this module only does what he decided. It never guesses a `note-status`, never
proposes a Hopsakee.Decimal, and never moves a file.

Two rules shape every function here:

- **The status vocabulary lives in the vault, not in this code.** `load_note_statuses`
  reads `Extra/metadata/notestatus-classes.md` so the picker can never drift from the
  eight values Jelle actually uses.
- **This pipeline writes exactly one key, and that key is hd.** No `hd-domain`, no
  `hd-proposed`, no `hoggle-status`. The value is refined in place: `d6` here, then
  `d6.75` once a category is picked.

In [ ]:
#| default_exp inbox

In [ ]:
#| export
import json
import re
import subprocess
from dataclasses import dataclass, field
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

## Paths and vocabulary

Everything authoritative is read from disk. `hd-structure.json` is rebuilt nightly at 02:00
by step 2.5 of `run-braincave-nightly.sh`, so it is always in lockstep with the vault's real
folder structure — which is exactly why this module reads it instead of the static
`config/d*.json` files `filesorter` uses for the Downloads tree.

In [ ]:
#| export
VAULT = Path.home() / "Obsidian" / "Braincave"
INBOX = VAULT / "0_Inbox"

#: Authoritative Hopsakee.Decimal structure. Rebuilt nightly; never hand-edited.
HD_STRUCTURE = Path.home() / ".claude" / "skills" / "_HD" / "hd-structure.json"

#: Canonical `note-status` vocabulary, owned by the vault.
NOTESTATUS_FILE = VAULT / "Extra" / "metadata" / "notestatus-classes.md"

LEDGER_DIR = Path.home() / ".claude" / "PAI" / "MEMORY" / "LEARNING"
LEDGER = LEDGER_DIR / "hd-choices.jsonl"
PENDING_DELETES = LEDGER_DIR / "inbox-pending-deletes.jsonl"
LEDGER_V = 1

#: Jelle's processing order (decided 2026-07-28). Small, high-signal folders first;
#: the 847-note NoteStation dump last. Folders not listed sort after these.
INBOX_ORDER = [
    "Phone", "Hoggle", "NewNotes", "Notes", "Readwise",
    "Snipd", "Atlas", "Sources", "Omnivore", "zz_NoteStationDumb",
]

#: `d0 Didymus` is a generated read-only mirror of `PAI/USER/` — no inbox note is ever
#: filed there, so it must not appear in the domain picker.
EXCLUDED_DOMAINS = {"d0"}

#: Frontmatter keys worth showing in the preview header. Everything else is noise
#: when the question is only "keep, delete, or too hard?".
PREVIEW_KEYS = ("hd", "note-status", "tags", "source", "created", "author", "url")

FM_DELIM = "---"
TODO_TASK = "- [ ] Organise this note [15m]"

## Hopsakee.Decimal structure

`hd-structure.json` uses `id` for the domain key (`d6`), **not** `domain_id` — so
`filesorter.domain.JDDomain` cannot parse it (it would derive `domain_id` from the first
word of `name` and produce `Programming`). Hence a dedicated loader.

In [ ]:
#| export
@dataclass
class HDArea:
    """A Hopsakee.Decimal area, e.g. `70-79 LLM en Generative AI`."""
    id: str
    name: str
    categories: list[dict] = field(default_factory=list)

    @property
    def hd_digit(self) -> str:
        """The area's `hd:` digit — `70-79` → `7`, so the area itself is `d6.7`.

        An area id is a decade range, and its digit is that range with the trailing
        unit dropped — which is exactly the prefix every category inside it shares
        (`70-79` holds `70`..`79`). So `d6.7` is simply a truncated `d6.75`: an area
        needs no syntax of its own, it is a shorter `hd:` value.
        """
        start = self.id.split("-")[0]
        return start[:-1] if len(start) > 1 else start

    @property
    def label(self) -> str: return f"{self.id} {self.name}"


@dataclass
class HDDomain:
    """A Hopsakee.Decimal domain, e.g. `d6 Programming en Development`."""
    id: str
    name: str
    areas: list[HDArea] = field(default_factory=list)

    @property
    def label(self) -> str: return f"{self.id} {self.name}"

    def hd(self, area: HDArea | None = None) -> str:
        """The `hd:` value for this domain, optionally narrowed to one of its areas."""
        return f"{self.id}.{area.hd_digit}" if area is not None else self.id


def load_hd_domains(path: Path = HD_STRUCTURE, exclude: set[str] = EXCLUDED_DOMAINS) -> list[HDDomain]:
    """Load domains from the authoritative structure cache, dropping generated mirrors."""
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    domains = []
    for d in data.get("domains", []) or []:
        did = (d.get("id") or "").strip()
        if not did or did in exclude: continue
        areas = [
            HDArea(id=(a.get("id") or ""), name=(a.get("name") or ""), categories=(a.get("categories") or []))
            for a in (d.get("areas") or [])
        ]
        domains.append(HDDomain(id=did, name=(d.get("name") or ""), areas=areas))
    return sorted(domains, key=lambda d: d.id)


def hd_structure_age_days(path: Path = HD_STRUCTURE) -> float:
    """Days since the structure cache was regenerated.

    `_BRAINCAVE_INBOX_SORT` aborts above 14 days because a stale tree can name categories
    that no longer exist. Triage only writes a *domain*, so it is far more tolerant — but
    the caller should still warn.
    """
    return (datetime.now().timestamp() - Path(path).stat().st_mtime) / 86400

In [ ]:
#| export
def load_note_statuses(path: Path = NOTESTATUS_FILE) -> list[str]:
    """Read the canonical `note-status` values from the vault, in file order.

    Deliberately not hardcoded: Jelle owns this list, and a copy here would silently
    drift the moment he adds a status.
    """
    out = []
    for raw in Path(path).read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith(("#", ">", "-", "*", "|")): continue
        out.append(line)
    return out

## Resolving an `hd:` to a folder

An `hd:` value is a *number*, not a path — `d1.4` is not a folder called `d1.4`, it is the
Hopsakee Decimal for `d1 Prive/40-49 Weten hoe men leeft`. The nightly `hd-structure.json`
already carries the answer: every **category** has a `folder_path`. Areas and domains do not,
but they do not need to — the vault tree is `domain/area/category`, so an area folder is any
of its categories' parent and a domain folder is their grandparent.

So every precision level resolves to a real directory, and the mover never has to guess.

In [ ]:
#| export
def load_hd_folders(path: Path = HD_STRUCTURE) -> dict[str, Path]:
    """Map every `hd:` value to its vault-relative folder, at all precisions.

    Returns e.g. `{"d1": "d1 Prive", "d1.4": "d1 Prive/40-49 …", "d1.44": "d1 Prive/40-49 …/44 …"}`.
    Categories come straight from `folder_path`; area and domain folders are derived from a
    category's parents. An area holding no categories falls back to globbing the domain folder
    for a directory starting with the area id.
    """
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    vault = Path(data.get("vault_path") or VAULT)
    folders: dict[str, Path] = {}

    for domain in data.get("domains", []) or []:
        did = (domain.get("id") or "").strip()
        if not did: continue
        for area in domain.get("areas", []) or []:
            aid = (area.get("id") or "").strip()
            area_digit = aid.split("-")[0][:-1] if len(aid.split("-")[0]) > 1 else aid
            cat_folder = None
            for cat in area.get("categories", []) or []:
                raw = cat.get("folder_path")
                if not raw: continue
                cat_folder = Path(raw)
                folders[f"{did}.{cat['id']}"] = cat_folder
            if cat_folder is not None:
                folders.setdefault(f"{did}.{area_digit}", cat_folder.parent)
                folders.setdefault(did, cat_folder.parent.parent)
            elif aid:
                # Area with no categories: find its directory under the domain folder.
                domain_folder = folders.get(did)
                if domain_folder is not None:
                    hit = next((d for d in sorted((vault / domain_folder).glob(f"{aid}*")) if d.is_dir()), None)
                    if hit is not None: folders.setdefault(f"{did}.{area_digit}", hit.relative_to(vault))
    return folders


def category_groups(hd_category, folders=None, vault=VAULT) -> dict[str, Path]:
    """Existing id groups inside a category, keyed by their complete `hd:`.

    An item id is not a note, it is a *group* — a named subfolder like `Hoggle_d6.75.04` or
    `; PAI_d6.75.01`. All notes about one project, or one book, share that folder and that id.
    The label carries the meaning, the suffix carries the number.
    """
    cat_folder = _resolve_structural(hd_category, folders, vault)
    if cat_folder is None or not cat_folder.is_dir(): return {}
    dom, num = hd_category.strip().split(".")[:2]
    # The vault names group folders four ways, and all four are legitimate:
    #   Hoggle_d6.75.04            suffix, with domain
    #   Isala_15.03                suffix, domain implied by location
    #   =! d5.43.49 Strategische…  leading, with domain (these sort projects by id on purpose)
    #   41.04 ADHD behandeltraject leading, domain implied
    # Matching only the first form made 60 of 224 groups invisible, which is what caused a
    # duplicate folder to be created for d1.41.04 on 2026-07-30.
    pattern = re.compile(rf"(?:^|[^\d])(?:{re.escape(dom)}\.)?{re.escape(num)}\.(\d+)(?:$|[^\d])")
    found: dict[str, list[Path]] = {}
    for child in sorted(cat_folder.iterdir()):
        if not child.is_dir(): continue
        match = pattern.search(child.name)
        if match: found.setdefault(f"{dom}.{num}.{match.group(1)}", []).append(child)
    # An id owned by two folders is ambiguous, and picking one alphabetically would file
    # notes into whichever sorted first — so it is excluded here and reported instead.
    return {hd: paths[0] for hd, paths in found.items() if len(paths) == 1}


def duplicate_groups(hd_category, folders=None, vault=VAULT) -> dict[str, list[Path]]:
    """Ids claimed by more than one folder in a category. These must be merged by hand."""
    cat_folder = _resolve_structural(hd_category, folders, vault)
    if cat_folder is None or not cat_folder.is_dir(): return {}
    dom, num = hd_category.strip().split(".")[:2]
    pattern = re.compile(rf"(?:^|[^\d])(?:{re.escape(dom)}\.)?{re.escape(num)}\.(\d+)(?:$|[^\d])")
    found: dict[str, list[Path]] = {}
    for child in sorted(cat_folder.iterdir()):
        if not child.is_dir(): continue
        match = pattern.search(child.name)
        if match: found.setdefault(f"{dom}.{num}.{match.group(1)}", []).append(child)
    return {hd: paths for hd, paths in found.items() if len(paths) > 1}


def mislabelled_groups(hd_category, folders=None, vault=VAULT) -> list[tuple[Path, str]]:
    """Group folders whose id contradicts where they sit — a data error, not a case to handle.

    Real example: `=! Fable projects 2026_d5.75.07` lives inside `d6.75` and its notes all say
    `hd: d6.75.07`, so the folder name's `d5` is a typo. Reporting beats accommodating; silently
    matching such a folder would file notes by a number the vault disagrees with.
    """
    cat_folder = _resolve_structural(hd_category, folders, vault)
    if cat_folder is None or not cat_folder.is_dir(): return []
    out = []
    for child in sorted(p for p in cat_folder.iterdir() if p.is_dir()):
        _, sep, suffix = child.name.rpartition("_")
        if sep and is_complete_hd(suffix) and not suffix.startswith(hd_category.strip() + "."):
            out.append((child, suffix))
    return out


def group_label(folder: Path) -> str:
    """The human part of an id-group folder name: `Hoggle_d6.75.04` gives `Hoggle`."""
    head, sep, _ = folder.name.rpartition("_")
    return (head if sep else folder.name).strip()


def _resolve_structural(value, folders, vault) -> Path | None:
    """Resolve a domain / area / category value from the nightly structure map."""
    folders = load_hd_folders() if folders is None else folders
    parts = value.strip().split(".")
    for stop in range(len(parts), 0, -1):
        key = ".".join(parts[:stop])
        if key in folders: return vault / folders[key]
    return None


def hd_to_folder(hd: str, folders=None, vault: Path = VAULT) -> Path | None:
    """Absolute destination folder for an `hd:` value.

    Domain, area and category resolve from the nightly structure. An **item-level** value
    resolves to its id-group *subfolder* inside the category: `d6.75.04` is
    `.../75 Agents en tool use/Hoggle_d6.75.04`, not the category folder. When no such group
    exists yet the answer is `None`, deliberately — that means a new group has to be created
    and only Jelle can name it. Falling back to the category folder would silently misfile.
    """
    if not hd or not hd.strip(): return None
    value = hd.strip()
    if is_complete_hd(value):
        category = ".".join(value.split(".")[:2])
        return category_groups(category, folders, vault).get(value)
    return _resolve_structural(value, folders, vault)

## Reading a note

The parser is line-based and keeps the original lines verbatim. That is a hard requirement:
these are 2657 hand-written notes, and a PyYAML round-trip would reorder keys, restyle
values and rewrite quoting across the whole inbox.

In [ ]:
#| export
@dataclass
class Note:
    """A parsed note that remembers its exact original lines."""
    path: Path
    lines: list[str]
    fm_start: int | None = None
    fm_end: int | None = None

    @property
    def has_frontmatter(self) -> bool:
        return self.fm_start is not None and self.fm_end is not None

    @property
    def fm_lines(self) -> list[str]:
        return self.lines[self.fm_start + 1:self.fm_end] if self.has_frontmatter else []

    @property
    def body_lines(self) -> list[str]:
        return self.lines[self.fm_end + 1:] if self.has_frontmatter else self.lines

    def get(self, key: str, default: str | None = None) -> str | None:
        """Value of a top-level frontmatter key, or `default` when absent/empty."""
        pat = re.compile(rf"^{re.escape(key)}:\s*(.*)$")
        for line in self.fm_lines:
            m = pat.match(line)
            if m: return m.group(1).strip() or default
        return default


def parse_note(path: Path) -> Note:
    """Parse a markdown note, locating its frontmatter fence if it has one."""
    p = Path(path)
    lines = p.read_text(encoding="utf-8", errors="replace").splitlines()
    fm_start = fm_end = None
    if lines and lines[0].strip() == FM_DELIM:
        for i, line in enumerate(lines[1:], start=1):
            if line.strip() == FM_DELIM:
                fm_start, fm_end = 0, i
                break
    return Note(path=p, lines=lines, fm_start=fm_start, fm_end=fm_end)

In [ ]:
#| export
def note_preview(note: Note, n: int = 20) -> tuple[list[str], list[str]]:
    """Return `(header, body)` — interesting frontmatter keys, then `n` non-empty body lines.

    Splitting these matters: a raw first-20-lines preview of a vault note shows only YAML,
    which tells you nothing about what the note is.
    """
    header = [f"{k}: {v}" for k in PREVIEW_KEYS if (v := note.get(k))]
    body: list[str] = []
    for line in note.body_lines:
        s = line.strip()
        if s and s != FM_DELIM:
            body.append(s)
            if len(body) >= n: break
    return header, body

## Backlinks

The single most decision-relevant signal, and the one neither `filesorter` nor
`_BRAINCAVE_INBOX_SORT` surfaces: **a note with inbound links is connected, so deleting it
breaks something.** Counting is one ripgrep pass over the vault at startup rather than a
grep per note, which would be 2657 subprocesses.

In [ ]:
#| export
WIKILINK = re.compile(r"\[\[([^\]|#]+)")


def build_backlink_index(vault: Path = VAULT, timeout: int = 120) -> dict[str, int]:
    """Count inbound wikilink targets across the whole vault in a single pass.

    Keyed on the link target's final path segment, which is how Obsidian resolves
    `[[Some Note]]`. Falls back to pure Python when ripgrep is unavailable.
    """
    counts: dict[str, int] = {}

    def tally(text: str) -> None:
        for target in WIKILINK.findall(text):
            key = target.split("/")[-1].strip()
            if key: counts[key] = counts.get(key, 0) + 1

    try:
        res = subprocess.run(
            ["rg", "-oI", "--no-filename", r"\[\[[^\]]+\]\]", "-g", "*.md", str(vault)],
            capture_output=True, text=True, timeout=timeout,
        )
        # rg exits 1 on "no matches", which is not an error here.
        if res.returncode in (0, 1):
            tally(res.stdout)
            return counts
    except Exception:
        pass

    for p in Path(vault).rglob("*.md"):
        try: tally(p.read_text(encoding="utf-8", errors="replace"))
        except Exception: continue
    return counts


def backlink_count(index: dict[str, int], path: Path) -> int:
    """Inbound link count for `path`, from a prebuilt index."""
    return index.get(Path(path).stem, 0)

In [ ]:
#| export
def list_candidates(inbox: Path = INBOX, order: list[str] = INBOX_ORDER) -> list[Path]:
    """Every inbox note, in Jelle's processing order then alphabetically within a folder."""
    inbox = Path(inbox)
    rank = {name: i for i, name in enumerate(order)}
    notes = [p for p in inbox.rglob("*.md") if not p.name.startswith(".")]

    def sort_key(p: Path):
        parts = p.relative_to(inbox).parts
        top = parts[0] if len(parts) > 1 else ""
        return (rank.get(top, len(order)), str(p).lower())

    return sorted(notes, key=sort_key)

## What "finished" means: only a complete number leaves the inbox

A note moves **once**, to its final place, and only when its `hd:` is complete — domain, area,
category and item id. `d1.53.12` qualifies. `d1`, `d1.5` and `d1.53` do not.

Note that a complete number has three dot-separated parts, not four: the area is not a segment
of its own, it is the first digit of the category. `d1.53.12` is domain `d1`, area `5`,
category `53`, item `12`.

That gives the pipeline two queues, both derived from the notes themselves:

- **`needs_proposal`** — Jelle kept it and narrowed it as far as he was confident, so its `hd:`
  is still partial. This is what the nightly proposer reads.
- **`ready_to_file`** — the `hd:` is complete, so the mover may file it. Nothing else is eligible,
  ever.

The item id is allocated from `max_hd_id`, which the nightly already stores per category — ids are
not contiguous (one category has 7 files and a max of 98), so the next free id is always max + 1.

In [ ]:
#| export
#: A complete Hopsakee Decimal: domain, two-digit category, item id. The area lives in the
#: category's first digit, so a complete number has three parts — `d1.53.12`.
COMPLETE_HD = re.compile(r"^d\d+\.\d{2}\.\d+$")


def is_complete_hd(hd: str | None) -> bool:
    """True only for a full `domain.category.id` — the one form allowed to leave the inbox."""
    return bool(hd and COMPLETE_HD.match(hd.strip()))


def next_item_id(hd_category: str, path: Path = HD_STRUCTURE) -> str | None:
    """Next free item id in a category, from the nightly `max_hd_id`, zero-padded to two.

    Ids are not contiguous, so the next free one is `max_hd_id + 1` rather than a count.
    """
    parts = hd_category.strip().split(".")
    if len(parts) < 2: return None
    domain_id, category_id = parts[0], parts[1]
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    for domain in data.get("domains", []) or []:
        if (domain.get("id") or "") != domain_id: continue
        for area in domain.get("areas", []) or []:
            for cat in area.get("categories", []) or []:
                if str(cat.get("id")) == category_id:
                    return f"{int(cat.get('max_hd_id') or 0) + 1:02d}"
    return None


def complete_hd(hd_category: str, path: Path = HD_STRUCTURE) -> str | None:
    """Turn a category-level value into a complete one: `d1.53` → `d1.53.97`."""
    item = next_item_id(hd_category, path)
    if item is None: return None
    candidate = f"{hd_category.strip()}.{item}"
    return candidate if is_complete_hd(candidate) else None


def needs_proposal(inbox: Path = INBOX, order: list[str] = INBOX_ORDER,
                   ledger: Path = LEDGER) -> list[tuple[Path, str]]:
    """Kept notes whose `hd:` is still partial — the nightly proposer's input queue."""
    decided = decided_notes(ledger)
    out = []
    for note_path in list_candidates(inbox, order):
        if note_path not in decided: continue
        hd = parse_note(note_path).get("hd")
        if hd and not is_complete_hd(hd): out.append((note_path, hd.strip()))
    return out


def ready_to_file(inbox: Path = INBOX, order: list[str] = INBOX_ORDER) -> list[tuple[Path, str]]:
    """Inbox notes carrying a complete `hd:` — the only notes the mover may ever touch."""
    out = []
    for note_path in list_candidates(inbox, order):
        hd = parse_note(note_path).get("hd")
        if is_complete_hd(hd): out.append((note_path, hd.strip()))
    return out

## Resuming where you stopped

The whole point of a fifteen-minute session is that it does not have to be a *complete* session.
Every verdict is on disk the moment it is made — `set_frontmatter` writes per note, the delete
queue is an append-only JSONL — so quitting after six notes loses nothing. What was missing is the
other half: the next run has to *skip those six*, or a short session just replays itself forever.

The ledger is the record of what was decided, so it is also the resume marker. `skip` deliberately
does not count as a decision: skipping means "not now", so the note returns next session. That is
the entire difference between `s` and `t`.

In [ ]:
#| export
#: Ledger events that mean "this note has a standing verdict".
DECIDED_EVENTS = {"keep", "delete_queued", "delete_applied", "todo"}


def decided_notes(ledger: Path = LEDGER) -> set[Path]:
    """Notes carrying a standing verdict, replayed in order from the ledger.

    An `undo` clears the note again, so undoing something genuinely puts it back in the queue.
    """
    p = Path(ledger)
    if not p.exists(): return set()
    decided: set[Path] = set()
    for line in p.read_text(encoding="utf-8").splitlines():
        if not line.strip(): continue
        try: record = json.loads(line)
        except json.JSONDecodeError: continue
        note = record.get("note")
        if not note: continue
        event = record.get("event_type")
        if event in DECIDED_EVENTS: decided.add(Path(note))
        elif event == "undo": decided.discard(Path(note))
    return decided


def pending_candidates(inbox: Path = INBOX, order: list[str] = INBOX_ORDER,
                       ledger: Path = LEDGER) -> tuple[list[Path], int]:
    """Inbox notes still awaiting a verdict, and how many are already decided."""
    decided = decided_notes(ledger)
    everything = list_candidates(inbox, order)
    pending = [p for p in everything if p not in decided]
    return pending, len(everything) - len(pending)

## Writing a verdict

`set_frontmatter` is the only writer, and it edits in place at an existing path — which is
the one vault mutation allowed outside the `obsidian` CLI, because no wikilink can break
when neither the filename nor the folder changes. Anything that *moves* a note stays the
CLI's job.

In [ ]:
#| export
def set_frontmatter(path: Path, updates: dict[str, str]) -> None:
    """Set frontmatter keys in place, preserving every other line verbatim.

    Existing keys are replaced where they sit; new keys are appended just before the closing
    fence; a note with no frontmatter gets a fresh block. Never reorders, never reformats.
    """
    p = Path(path)
    note = parse_note(p)
    original = p.read_text(encoding="utf-8", errors="replace")

    if not note.has_frontmatter:
        block = [FM_DELIM, *[f"{k}: {v}" for k, v in updates.items()], FM_DELIM]
        lines = block + note.lines
    else:
        lines = list(note.lines)
        fm_end = note.fm_end
        for key, value in updates.items():
            pat = re.compile(rf"^{re.escape(key)}:\s*(.*)$")
            hit = next((i for i in range(note.fm_start + 1, fm_end) if pat.match(lines[i])), None)
            if hit is not None:
                lines[hit] = f"{key}: {value}"
            else:
                lines.insert(fm_end, f"{key}: {value}")
                fm_end += 1

    trailing = "\n" if original.endswith("\n") or not original else ""
    p.write_text("\n".join(lines) + trailing, encoding="utf-8")


def append_task_line(path: Path, when: date | None = None, text: str = TODO_TASK) -> str:
    """Append a `_DAILY_PLAN`-schedulable task line to the note itself.

    `_DAILY_PLAN` schedules work by putting `⏳ YYYY-MM-DD` on a task line inside its source
    note, so a "this one is too hard right now" verdict becomes a task that lives in the note
    it is about — no separate task file, no wikilink plumbing. Defaults to tomorrow.
    """
    when = when or (date.today() + timedelta(days=1))
    line = f"{text} ⏳ {when.isoformat()}"
    p = Path(path)
    current = p.read_text(encoding="utf-8", errors="replace")
    sep = "" if current.endswith("\n") or not current else "\n"
    p.write_text(f"{current}{sep}{line}\n", encoding="utf-8")
    return line

## The ledger

Every verdict is logged. The pick-vs-proposal comparison this enables is the point of the
whole pipeline: a category Jelle keeps overriding is a category whose *description* is wrong,
and that description lives in the vault MOC where the next nightly rebuild will pick it up.

Field order (`timestamp`, `v`, `tool`, `event_type`) follows the PAI observability rule so
cross-file `jq` consumers work without special-casing this file.

In [ ]:
#| export
def _append_jsonl(path: Path, record: dict) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def log_event(event_type: str, note: Path, **payload) -> dict:
    """Append one triage event to the ledger and return the record."""
    record = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "v": LEDGER_V,
        "tool": "inbox-triage",
        "event_type": event_type,
        "note": str(Path(note)),
    }
    record.update(payload)
    _append_jsonl(LEDGER, record)
    return record


def queue_delete(path: Path) -> dict:
    """Record an intent to delete, without deleting.

    Deleting is the one verdict the `obsidian` CLI must own — it routes to the OS trash and
    keeps the vault's link graph consistent — and that CLI needs the desktop app running,
    which triage deliberately does not. Queuing also means a session of 200 `d` presses stays
    reviewable before anything is actually gone.
    """
    record = log_event("delete_queued", path)
    _append_jsonl(PENDING_DELETES, {**record, "event_type": "delete_pending"})
    return record


def _unqueue_delete(path: Path) -> None:
    """Drop `path` from the pending-delete queue (used by undo)."""
    p = Path(PENDING_DELETES)
    if not p.exists(): return
    target = str(Path(path))
    kept = []
    for line in p.read_text(encoding="utf-8").splitlines():
        if not line.strip(): continue
        try: keep = json.loads(line).get("note") != target
        except json.JSONDecodeError: keep = True
        if keep: kept.append(line)
    p.write_text("\n".join(kept) + ("\n" if kept else ""), encoding="utf-8")


def pending_deletes() -> list[Path]:
    """Notes currently queued for deletion."""
    p = Path(PENDING_DELETES)
    if not p.exists(): return []
    out = []
    for line in p.read_text(encoding="utf-8").splitlines():
        if not line.strip(): continue
        try: out.append(Path(json.loads(line)["note"]))
        except (json.JSONDecodeError, KeyError): continue
    return out

## Never coarsen an existing `hd:`

`hd:` is refined left to right — `d6` → `d6.7` → `d6.75` → `d6.75.04` — so a note that already
carries a category is *ahead* of what triage can offer, and writing the area over it would throw
precision away. 436 inbox notes already have an `hd:`, so this is the common case, not an edge
case. Triage therefore keeps the finer value and only sets the status.

A value that *conflicts* (existing `d1.2`, Jelle picks `d6`) is different: he is looking at the
note right now, so his live choice wins — but the status bar says what it replaced.

In [ ]:
#| export
def hd_precision(value: str | None) -> int:
    """How specific an `hd:` value is: 0 none, 1 domain, 2 area, 3 category, 4+ item.

    `d6` → 1, `d6.7` → 2, `d6.75` → 3, `d6.75.04` → 4. Area and category are both a single
    dot-part, told apart by digit count, which is exactly why `d6.7` is a prefix of `d6.75`.
    """
    if not value: return 0
    parts = value.strip().split(".")
    if len(parts) == 1: return 1
    level = 1 + min(len(parts[1]), 2)
    return level + (len(parts) - 2)


def refines(finer: str | None, coarser: str | None) -> bool:
    """True when `finer` is a strictly more specific version of the same `hd:` path."""
    if not finer or not coarser: return False
    return finer.startswith(coarser) and hd_precision(finer) > hd_precision(coarser)


def resolve_hd(existing: str | None, chosen: str) -> tuple[str, str]:
    """Decide the `hd:` to write. Returns `(value, note)` where note explains a non-write."""
    if refines(existing, chosen):
        return existing, f"kept existing {existing} (finer than {chosen})"
    if existing and existing.strip() and not chosen.startswith(existing.strip()):
        return chosen, f"replaced {existing.strip()}"
    return chosen, ""

## Applying verdicts

`Triage` snapshots a note's bytes before mutating it, so undo is an exact restore rather than
a best-effort inverse edit.

In [ ]:
#| export
@dataclass
class Action:
    """One applied verdict, with enough state to undo it exactly."""
    note: Path
    kind: str
    before: str | None = None
    delete_queued: bool = False


class Triage:
    """Applies Jelle's verdicts and keeps a per-session undo stack."""

    def __init__(self):
        self.stack: list[Action] = []
        self.counts: dict[str, int] = {}

    def _snapshot(self, path: Path) -> str:
        return Path(path).read_text(encoding="utf-8", errors="replace")

    def _push(self, action: Action) -> None:
        self.stack.append(action)
        self.counts[action.kind] = self.counts.get(action.kind, 0) + 1

    def keep(self, path: Path, status: str, hd: str) -> str:
        """Record a keeper: Jelle's chosen `note-status`, plus the `hd:` he narrowed to.

        Never coarsens an `hd:` the note already carries — see `resolve_hd`.
        """
        before = self._snapshot(path)
        existing = parse_note(path).get("hd")
        value, note = resolve_hd(existing, hd)
        set_frontmatter(path, {"note-status": status, "hd": value})
        log_event("keep", path, note_status=status, hd=value, chosen=hd,
                  existing_hd=existing, kept_existing=bool(note.startswith("kept")))
        self._push(Action(Path(path), "keep", before))
        return f"{status} · hd: {value}" + (f" · {note}" if note else "")

    def delete(self, path: Path) -> str:
        queue_delete(path)
        self._push(Action(Path(path), "delete", None, delete_queued=True))
        return "queued for delete"

    def todo(self, path: Path, when: date | None = None) -> str:
        before = self._snapshot(path)
        line = append_task_line(path, when)
        log_event("todo", path, task=line)
        self._push(Action(Path(path), "todo", before))
        return line

    def skip(self, path: Path) -> str:
        log_event("skip", path)
        self._push(Action(Path(path), "skip", None))
        return "skipped"

    def can_undo(self) -> bool:
        return bool(self.stack)

    def undo(self) -> Action | None:
        """Restore the last note byte-for-byte and unqueue any delete it had."""
        if not self.stack: return None
        action = self.stack.pop()
        if action.before is not None:
            action.note.write_text(action.before, encoding="utf-8")
        if action.delete_queued:
            _unqueue_delete(action.note)
        log_event("undo", action.note, undone=action.kind)
        self.counts[action.kind] = max(0, self.counts.get(action.kind, 1) - 1)
        return action

    def summary(self) -> str:
        if not self.counts: return "nothing triaged"
        return " | ".join(f"{n} {kind}" for kind, n in sorted(self.counts.items()))

## The `obsidian` CLI edges

Two operations must go through the CLI: opening a note (needs the running app anyway) and
draining the delete queue. `drain_deletes` defaults to a dry run — deleting 200 notes should
never be a side effect of a typo.

In [ ]:
#| export
def obsidian_relpath(path: Path, vault: Path = VAULT) -> str:
    """Vault-relative path, which is what `obsidian path=` expects."""
    return str(Path(path).resolve().relative_to(Path(vault).resolve()))


def obsidian_open(path: Path, vault_name: str = "Braincave") -> subprocess.CompletedProcess:
    """Open a note in the Obsidian desktop app."""
    return subprocess.run(
        ["obsidian", "open", f"path={obsidian_relpath(path)}", f"vault={vault_name}"],
        capture_output=True, text=True, timeout=30,
    )


def drain_deletes(dry_run: bool = True, vault_name: str = "Braincave") -> list[tuple[Path, str]]:
    """Apply queued deletes via `obsidian delete` (OS trash — recoverable).

    Never passes `permanent`. Requires the Obsidian desktop app to be running. Returns
    `(note, outcome)` per queued note; on a real run the queue is rewritten to hold only
    the failures, so re-running retries exactly those.
    """
    results: list[tuple[Path, str]] = []
    failed: list[Path] = []
    for note in pending_deletes():
        if dry_run:
            results.append((note, "would delete"))
            continue
        try:
            res = subprocess.run(
                ["obsidian", "delete", f"path={obsidian_relpath(note)}", f"vault={vault_name}"],
                capture_output=True, text=True, timeout=30,
            )
            ok = res.returncode == 0
            results.append((note, "deleted" if ok else f"failed: {(res.stderr or res.stdout).strip()[:80]}"))
            if ok: log_event("delete_applied", note)
            else: failed.append(note)
        except Exception as e:
            results.append((note, f"failed: {e}"))
            failed.append(note)

    if not dry_run:
        p = Path(PENDING_DELETES)
        p.write_text("", encoding="utf-8")
        for note in failed: queue_delete(note)
    return results

## Where a note actually goes

Three cases, and only the first can proceed unattended.

1. **The id group already exists as a folder** — move the note in. Done.
2. **The id exists only as a single note** (232 of them in the vault, e.g.
   `! Tools om te denken_d1.43.07.md`). Then a folder named after that existing note is created
   and *both* notes move into it — Jelle's rule, and it needs no new label because the existing
   note already carries one.
3. **Nothing anywhere uses that id** — a genuinely new group. That needs a label, and only Jelle
   can supply it, so the planner stops and says so.

New group folders are always `Label_hd` with an **underscore**. The vault had 13 names using a
space instead; those were corrected on 2026-07-29 so the convention is now uniform.

One deliberate non-case: `d5` area `50-59` is absent from `hd-structure.json` because it is
*vervallen*, not because BuildCache misses it. Notes pointing there resolve to nothing on purpose
— do not "fix" the cache.

In [ ]:
#| export
def group_folder_name(label: str, hd: str) -> str:
    """Name for a new id-group folder. Always an underscore before the number."""
    return f"{label.strip()}_{hd.strip()}"


@dataclass
class FilePlan:
    """What has to happen for one note to reach its final place."""
    note: Path
    hd: str
    kind: str                      # into_group | promote_note | needs_label | no_category
    destination: Path | None = None
    promote: Path | None = None    # the existing single note that becomes a folder
    reason: str = ""

    @property
    def ready(self) -> bool:
        """True when the move needs no further input from Jelle."""
        return self.kind in {"into_group", "promote_note"}


def id_carrying_notes(vault: Path = VAULT, domains: tuple[str, ...] = ()) -> dict[str, Path]:
    """Filed notes whose own filename carries a complete id, keyed by that id."""
    found: dict[str, Path] = {}
    for folder in (domains or tuple(sorted({p.name for p in Path(vault).iterdir()
                                            if p.is_dir() and p.name.startswith("d")}))):
        root = Path(vault) / folder
        if not root.is_dir(): continue
        for note in root.rglob("*.md"):
            _, sep, suffix = note.stem.rpartition("_")
            if sep and is_complete_hd(suffix): found.setdefault(suffix, note)
    return found


def plan_filing(note: Path, hd: str, folders: dict[str, Path] | None = None,
                id_notes: dict[str, Path] | None = None, vault: Path = VAULT) -> FilePlan:
    """Decide where `note` goes, or say what is missing."""
    if not is_complete_hd(hd):
        return FilePlan(note, hd, "no_category", reason="hd: is not a complete number")
    folders = load_hd_folders() if folders is None else folders
    category = ".".join(hd.split(".")[:2])
    if _resolve_structural(category, folders, vault) is None:
        return FilePlan(note, hd, "no_category",
                        reason=f"category {category} is not in the structure (vervallen?)")

    existing = category_groups(category, folders, vault).get(hd)
    if existing is not None:
        return FilePlan(note, hd, "into_group", destination=existing)

    clashing = duplicate_groups(category, folders, vault).get(hd)
    if clashing:
        names = ", ".join(repr(c.name) for c in clashing)
        return FilePlan(note, hd, "ambiguous_group",
                        reason=f"{len(clashing)} folders claim {hd}: {names} — merge them first")

    id_notes = id_carrying_notes(vault) if id_notes is None else id_notes
    anchor = id_notes.get(hd)
    if anchor is not None and anchor != note:
        return FilePlan(note, hd, "promote_note", destination=anchor.parent / anchor.stem,
                        promote=anchor,
                        reason=f"id {hd} is currently the single note {anchor.name!r}")
    return FilePlan(note, hd, "needs_label",
                    reason=f"id {hd} is new — a group folder needs a label")

## Reading a proposal, and acting on it

Stage 3 is Jelle choosing; stage 4 is the move. Both are deterministic — the model already
did its work overnight and its output is sitting in the note body.

The mover is the only code in this pipeline that relocates anything, and it goes through the
`obsidian` CLI so wikilinks survive. It also **verifies the file actually arrived** rather than
trusting the exit code: the delete path taught us that a CLI can report success while nothing
happened, and a move that silently no-ops would leave a note claiming a home it never reached.

In [ ]:
#| export
#: A candidate line inside the proposal block: `- **d1.44.03** — Category · group`
PROPOSAL_RE = re.compile(r"^- \*\*(d\d+\.\d{2}\.\d+)\*\* — (.*)$", re.M)
BLOCK_START = "<!-- hoggle-hd-voorstel:start -->"
BLOCK_END = "<!-- hoggle-hd-voorstel:end -->"
BLOCK_RE = re.compile(re.escape(BLOCK_START) + r".*?" + re.escape(BLOCK_END), re.DOTALL)


def parse_proposals(path: Path) -> list[tuple[str, str]]:
    """`(hd, description)` candidates from a note's proposal block, in the order written."""
    text = Path(path).read_text(encoding="utf-8", errors="replace")
    match = BLOCK_RE.search(text)
    if not match: return []
    return [(hd, rest.strip()) for hd, rest in PROPOSAL_RE.findall(match.group(0))]


def notes_with_proposals(inbox: Path = INBOX, order: list[str] = INBOX_ORDER) -> list[Path]:
    """Inbox notes carrying a proposal block — stage 3's queue."""
    return [p for p in list_candidates(inbox, order)
            if BLOCK_START in p.read_text(encoding="utf-8", errors="replace")]


def apply_pick(path: Path, hd: str) -> str:
    """Promote a chosen candidate into `hd:` and drop the proposal block.

    Both edits are in place at an existing path, so no wikilink is at risk.
    """
    if not is_complete_hd(hd): raise ValueError(f"{hd!r} is not a complete number")
    set_frontmatter(path, {"hd": hd})
    text = Path(path).read_text(encoding="utf-8", errors="replace")
    Path(path).write_text(BLOCK_RE.sub("", text).rstrip() + "\n", encoding="utf-8")
    log_event("pick", path, hd=hd, proposed=[c for c, _ in parse_proposals(path)])
    return hd

In [ ]:
#| export
def _obsidian(*args: str, vault_name: str = "Braincave") -> subprocess.CompletedProcess:
    return subprocess.run(["obsidian", *args, f"vault={vault_name}"],
                          capture_output=True, text=True,
                          stdin=subprocess.DEVNULL, timeout=60)


def _move(src: Path, dest_dir: Path, vault: Path = VAULT) -> tuple[bool, str]:
    """Move one file into `dest_dir`, then confirm it is really there."""
    dest = dest_dir / src.name
    if dest.exists(): return False, f"target exists: {dest.name}"
    res = _obsidian("move", f"path={src.relative_to(vault)}", f"to={dest.relative_to(vault)}")
    if not dest.exists() or src.exists():
        detail = (res.stderr or res.stdout or "").strip()[:80]
        return False, f"move did not take effect{': ' + detail if detail else ''}"
    return True, str(dest.relative_to(vault))


def file_note(plan: "FilePlan", vault: Path = VAULT, dry_run: bool = True) -> tuple[bool, str]:
    """Execute one filing plan. Returns `(moved, detail)`.

    `into_group` moves the note into an existing group folder. `promote_note` first turns the
    lone anchor note into a folder named after itself — Jelle's rule — then moves both in.
    Anything needing a label, or pointing at a category the structure does not have, is refused.
    """
    if not plan.ready: return False, plan.reason or plan.kind
    if plan.kind == "into_group":
        if dry_run: return True, f"would move into {plan.destination.name}"
        ok, detail = _move(plan.note, plan.destination, vault)
        if ok: log_event("filed", plan.note, hd=plan.hd, kind=plan.kind,
                         destination=str(plan.destination))
        return ok, detail

    # promote_note: the id currently exists only as a single note.
    anchor, folder = plan.promote, plan.destination
    # Last-line guard. plan_filing already looked for a group, but it looks through one
    # matcher; if that matcher is ever wrong again, creating a second folder for an id that
    # already has one is the expensive mistake — it splits a project across two homes.
    existing = category_groups(".".join(plan.hd.split(".")[:2]), None, vault).get(plan.hd)
    if existing is not None:
        return False, f"refused: {existing.name!r} already holds {plan.hd}"
    if dry_run:
        return True, f"would create {folder.name}/ and move {anchor.name} + this note in"
    folder.mkdir(parents=True, exist_ok=True)
    ok, detail = _move(anchor, folder, vault)
    if not ok: return False, f"promoting {anchor.name} failed: {detail}"
    ok, detail = _move(plan.note, folder, vault)
    if ok: log_event("filed", plan.note, hd=plan.hd, kind=plan.kind, destination=str(folder))
    return ok, detail

## Smoke tests

These run against the real vault, read-only, so they double as a health check on the
contracts this module depends on.

In [ ]:
statuses = load_note_statuses()
print(statuses)
assert len(statuses) >= 5, "note-status vocabulary looks empty"
assert any("vervallen" in s for s in statuses)

In [ ]:
domains = load_hd_domains()
print([d.label for d in domains], f"| structure is {hd_structure_age_days():.1f} days old")
assert domains, "no domains loaded"
assert all(d.id != "d0" for d in domains), "d0 must never be an inbox destination"

The area digit has to be the real shared prefix of the categories inside it, or an
`hd: d6.7` would not be a truncation of `d6.75` and the whole no-new-syntax argument
collapses. So assert it against every area in the live structure.

In [ ]:
checked = 0
for d in domains:
    for a in d.areas:
        for cat in a.categories:
            assert str(cat["id"]).startswith(a.hd_digit),                 f"{d.id} area {a.id} (digit {a.hd_digit}) holds category {cat['id']}"
            checked += 1
        assert d.hd(a) == f"{d.id}.{a.hd_digit}"
print(f"{checked} categories all sit under their area digit")
print("examples:", [d.hd(d.areas[0]) for d in domains], "| domain-only:", [d.hd() for d in domains])
assert checked > 100

In [ ]:
cands = list_candidates()
print(f"{len(cands)} inbox notes; first three folders:",
      [c.relative_to(INBOX).parts[0] for c in cands[:3]])
assert cands, "no inbox candidates found"
assert cands[0].relative_to(INBOX).parts[0] == "Phone", "Phone must sort first"

In [ ]:
note = parse_note(cands[0])
header, body = note_preview(note)
print("HEADER:", header)
print("BODY  :", body[:5])
assert isinstance(header, list) and isinstance(body, list)

In [ ]:
folders = load_hd_folders()
print(f"{len(folders)} hd values resolve to a folder")
missing = [(k, v) for k, v in folders.items() if not (VAULT / v).is_dir()]
assert not missing, f"these hd values point at a non-existent folder: {missing[:5]}"

# Every precision level must resolve, and each must be inside the previous one.
for d in load_hd_domains():
    dom = hd_to_folder(d.id, folders)
    assert dom is not None and dom.is_dir(), f"{d.id} unresolved"
    for a in d.areas:
        area = hd_to_folder(d.hd(a), folders)
        assert area is not None and area.is_dir(), f"{d.hd(a)} ({a.id}) unresolved"
        assert area.parent == dom, f"{d.hd(a)} is not inside {dom}"
        for cat in a.categories:
            cf = hd_to_folder(f"{d.id}.{cat['id']}", folders)
            assert cf is not None and cf.is_dir(), f"{d.id}.{cat['id']} unresolved"
            assert cf.parent == area, f"{d.id}.{cat['id']} is not inside {area}"
print("domain ⊃ area ⊃ category holds for every entry")
print("  d1    ->", hd_to_folder("d1", folders).name)
print("  d1.4  ->", hd_to_folder("d1.4", folders).name)
print("  d1.44 ->", hd_to_folder("d1.44", folders).name)
groups = category_groups("d6.75", folders)
print(f"  d6.75 holds {len(groups)} id group(s):")
for item_hd, folder in sorted(groups.items()):
    print(f"     {item_hd}  {group_label(folder)!r}")
    assert hd_to_folder(item_hd, folders) == folder, f"{item_hd} must resolve to its own group"
    assert folder.parent == hd_to_folder("d6.75", folders), "a group sits inside its category"
assert hd_to_folder("d6.75.99", folders) is None, "an unknown id group must not fall back"
for folder, suffix in mislabelled_groups("d6.75", folders):
    print(f"  MISLABELLED: {folder.name!r} claims {suffix} but sits in d6.75")

In [ ]:
assert is_complete_hd("d1.53.12") and is_complete_hd("d6.12.19") and is_complete_hd("d6.75.04")
for partial in ["d1", "d1.5", "d1.53", "", None, "d1.5.12", "d1.533.12", "x1.53.12"]:
    assert not is_complete_hd(partial), f"{partial!r} must not count as complete"
assert hd_precision("d1.53.12") == 4
print("complete-vs-partial rules hold")

print("next free item in d1.53 ->", complete_hd("d1.53"))
assert complete_hd("d1.53") == f"d1.53.{next_item_id('d1.53')}"
assert is_complete_hd(complete_hd("d1.53"))
assert complete_hd("d1") is None, "a domain alone cannot be completed"

In [ ]:
# The two queues, against the live vault.
awaiting = needs_proposal()
filable = ready_to_file()
print(f"{len(awaiting)} kept note(s) awaiting a proposal, {len(filable)} ready to file")
for note_path, hd in awaiting[:5]: print(f"  needs proposal: {hd:<8} {note_path.name}")
for note_path, hd in filable[:5]: print(f"  ready to file : {hd:<12} {note_path.name}")
assert all(not is_complete_hd(hd) for _, hd in awaiting)
assert all(is_complete_hd(hd) for _, hd in filable), "the mover queue must be complete-only"
assert not ({p for p, _ in awaiting} & {p for p, _ in filable}), "a note is in one queue or the other"

In [ ]:
pending, already = pending_candidates()
print(f"{len(pending)} pending · {already} already decided")
assert len(pending) + already == len(list_candidates())
assert not (set(pending) & decided_notes()), "a decided note must never be offered again"

In [ ]:
assert [hd_precision(v) for v in [None, "", "d6", "d6.7", "d6.75", "d6.75.04"]] == [0, 0, 1, 2, 3, 4]
assert refines("d6.75", "d6.7") and refines("d6.7", "d6") and refines("d6.75.04", "d6.75")
assert not refines("d6.7", "d6.75"), "coarser must never count as a refinement"
assert not refines("d1.2", "d6"), "different domains are not refinements"
assert resolve_hd("d6.75", "d6.7")[0] == "d6.75", "must keep the finer existing value"
assert resolve_hd("d6.75", "d6")[0] == "d6.75"
assert resolve_hd(None, "d6.7")[0] == "d6.7"
assert resolve_hd("d1.2", "d6.7")[0] == "d6.7", "a conflicting live choice wins"
assert "replaced d1.2" in resolve_hd("d1.2", "d6.7")[1]
assert resolve_hd("d6.7", "d6.7")[0] == "d6.7"
print("hd precision + refinement rules hold")

In [ ]:
#| eval: false
# Round-trip check for the frontmatter writer, on a throwaway file outside the vault.
import tempfile
with tempfile.TemporaryDirectory() as td:
    p = Path(td) / "t.md"
    p.write_text("---\ntitle: Test\ntags:\n  - a\n---\n\nBody line\n", encoding="utf-8")
    set_frontmatter(p, {"note-status": "🌳done", "hd": "d6"})
    out = p.read_text(encoding="utf-8")
    print(out)
    assert "title: Test" in out and "  - a" in out, "existing frontmatter must survive verbatim"
    assert "note-status: 🌳done" in out and "hd: d6" in out
    assert out.rstrip().endswith("Body line")
    set_frontmatter(p, {"hd": "d6.75"})
    assert "hd: d6.75" in p.read_text(encoding="utf-8")
    assert p.read_text(encoding="utf-8").count("hd:") == 1, "must replace, not duplicate"
    print("frontmatter round-trip OK")